<a href="https://colab.research.google.com/github/Oct-o-Dev/DL/blob/main/Keras_HyperParameter_Tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import numpy as numpy
import pandas as pd

In [4]:
df = pd.read_csv('/content/diabetes.csv')

In [5]:
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72.0,35,169.5,33.6,0.627,50,1
1,1,85,66.0,29,102.5,26.6,0.351,31,0
2,8,183,64.0,32,169.5,23.3,0.672,32,1
3,1,89,66.0,23,94.0,28.1,0.167,21,0
4,0,137,40.0,35,168.0,43.1,2.288,33,1


In [6]:
df.corr()['Outcome']

,Outcome
Pregnancies,0.221898
Glucose,0.495990
BloodPressure,0.174469
SkinThickness,0.295138
Insulin,0.377081
BMI,0.315577
DiabetesPedigreeFunction,0.173844
Age,0.238356
Outcome,1.000000


In [7]:
X = df.iloc[:,:-1].values
y = df.iloc[:,-1].values

In [8]:
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
X = sc.fit_transform(X)

In [9]:
X.shape

(768, 8)

In [10]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=1)

In [53]:
import tensorflow
from tensorflow import keras
from keras.models import Sequential
from keras.layers import Dense,Dropout

In [12]:
from flax.nnx.training import optimizer
model = Sequential()
model.add(Dense(32,activation='relu',input_dim=8))
model.add(Dense(1,activation='sigmoid'))

model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [13]:
model.fit(X_train , y_train , batch_size=32 , epochs = 100 , validation_data=(X_test,y_test))

Epoch 1/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.2769 - loss: 0.9131 - val_accuracy: 0.3052 - val_loss: 0.8354
Epoch 2/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.3127 - loss: 0.7951 - val_accuracy: 0.4091 - val_loss: 0.7385
Epoch 3/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.5261 - loss: 0.7074 - val_accuracy: 0.6364 - val_loss: 0.6639
Epoch 4/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6531 - loss: 0.6443 - val_accuracy: 0.7013 - val_loss: 0.6092
Epoch 5/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7003 - loss: 0.5981 - val_accuracy: 0.7532 - val_loss: 0.5665
Epoch 6/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7199 - loss: 0.5627 - val_accuracy: 0.7792 - val_loss: 0.5330
Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7362 - loss: 0.5348 - val_accuracy: 0.8247 - val_loss: 0.5064
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7541 - loss: 0.5143 - val_accuracy: 0.8182 - 

In [14]:
# 1. How to select app. optimizer
# No. of nodes in a layer
# How to select no. of layer
# All in one model

In [15]:
pip install -U keras-tuner

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.4/129.4 kB 2.7 MB/s eta 0:00:00


In [16]:
import kerastuner as kt

/tmp/ipykernel_7053/1654478174.py:1: DeprecationWarning: `import kerastuner` is deprecated, please use `import keras_tuner`.
  import kerastuner as kt


In [17]:
def build_model(hp):

  model = Sequential()

  model.add(Dense(32,activation='relu',input_dim=8))
  model.add(Dense(1,activation='sigmoid'))

  optimizer = hp.Choice('optimizer' ,values =  ['adam' , 'sgd' , 'rmsprop' , 'adadelta'])

  model.compile(optimizer=optimizer , loss='binary_crossentropy', metrics =['accuracy'])

  return model

In [18]:
tuner = kt.RandomSearch(build_model , objective='val_accuracy' , max_trials=5)

In [19]:
tuner.search(X_train , y_train , epochs=5 , validation_data=(X_test , y_test))

Trial 3 Complete [00h 00m 04s]
val_accuracy: 0.8246753215789795

Best val_accuracy So Far: 0.8246753215789795
Total elapsed time: 00h 00m 15s


In [20]:
tuner.get_best_hyperparameters()[0].values

{'optimizer': 'rmsprop'}

In [21]:
model = tuner.get_best_models(num_models=1)[0]

/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 2 variables whereas the saved optimizer has 6 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [22]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 32)             │           288 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 321 (1.25 KB)

 Trainable params: 321 (1.25 KB)

 Non-trainable params: 0 (0.00 B)

In [23]:
model.fit(X_train , y_train , epochs=100 , initial_epoch = 6 , validation_data=(X_test , y_test))

Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.7801 - loss: 0.4605 - val_accuracy: 0.8312 - val_loss: 0.4248
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7801 - loss: 0.4463 - val_accuracy: 0.8247 - val_loss: 0.4138
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7801 - loss: 0.4381 - val_accuracy: 0.8247 - val_loss: 0.4054
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7883 - loss: 0.4317 - val_accuracy: 0.8247 - val_loss: 0.3983
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7948 - loss: 0.4264 - val_accuracy: 0.8312 - val_loss: 0.3942
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7964 - loss: 0.4225 - val_accuracy: 0.8377 - val_loss: 0.3908
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7997 - loss: 0.4185 - val_accuracy: 0.8442 - val_loss: 0.3869
Epoch 14/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7997 - loss: 0.4151 - val_accuracy: 0.83

In [24]:
def build_model(hp):

  model = Sequential()

  units = hp.Int('units' , min_value = 8, max_value = 128,step=8)
  model.add(Dense(units= units , activation='relu', input_dim= 8))
  model.add(Dense(1 , activation='sigmoid'))

  model.compile(optimizer='rmsprop' , loss='binary_crossentropy' , metrics=['accuracy'])

  return model

In [25]:
tuner = kt.RandomSearch(build_model , objective='val_accuracy' , max_trials=5 , directory='mydir')

In [26]:
tuner.search(X_train , y_train , epochs=5 , validation_data=(X_test , y_test))

Trial 5 Complete [00h 00m 02s]
val_accuracy: 0.6688311696052551

Best val_accuracy So Far: 0.8311688303947449
Total elapsed time: 00h 00m 11s


In [27]:
tuner.get_best_hyperparameters()[0].values

{'units': 128}

In [28]:
model = tuner.get_best_models(num_models = 1)[0]

In [29]:
model.fit(X_train , y_train , epochs=100 , initial_epoch=6 , validation_data=(X_test , y_test))

Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.7769 - loss: 0.4679 - val_accuracy: 0.8117 - val_loss: 0.4180
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7932 - loss: 0.4455 - val_accuracy: 0.8117 - val_loss: 0.4021
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7866 - loss: 0.4341 - val_accuracy: 0.8182 - val_loss: 0.3940
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7997 - loss: 0.4248 - val_accuracy: 0.8182 - val_loss: 0.3863
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7948 - loss: 0.4191 - val_accuracy: 0.8312 - val_loss: 0.3809
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7997 - loss: 0.4118 - val_accuracy: 0.8312 - val_loss: 0.3801
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8062 - loss: 0.4062 - val_accuracy: 0.8312 - val_loss: 0.3760
Epoch 14/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8078 - loss: 0.3999 - val_accuracy: 0.83

In [30]:
def build_model(hp):

  model = Sequential()

  model.add(Dense(32,activation='relu',input_dim=8))


  for i in range(hp.Int('num_layers' , min_value=1 , max_value=10)):
    model.add(Dense(72,activation='relu'))


  model.add(Dense(1,activation='sigmoid'))
  model.compile(optimizer='rmsprop' , loss='binary_crossentropy' , metrics=['accuracy'])

  return model

In [32]:
tuner = kt.RandomSearch(build_model , objective='val_accuracy' , max_trials = 3)

Reloading Tuner from ./untitled_project/tuner0.json


In [36]:
tuner.search(X_train , y_train , epochs=5 , validation_data=(X_test , y_test))

In [37]:
best_hp = tuner.get_best_hyperparameters()[0]
print(f'Best Hyperparameters: {best_hp.values}')

model = tuner.get_best_models(num_models=1)[0]
model.summary()

{'optimizer': 'rmsprop'}

In [58]:
def build_model(hp):
  model = Sequential()

  for i in range(hp.Int('num_layers', min_value=1, max_value=10)):
    if i == 0:
      model.add(Dense(
          units=hp.Int('units_' + str(i), min_value=8, max_value=128, step=8),
          activation=hp.Choice('activation_' + str(i), values=['relu', 'tanh', 'sigmoid']),
          input_dim=8
      ))
      model.add(Dropout(hp.Float('dropout_' + str(i), min_value=0.1, max_value=0.9, step=0.1)))
    else:
      model.add(Dense(
          units=hp.Int('units_' + str(i), min_value=8, max_value=128, step=8),
          activation=hp.Choice('activation_' + str(i), values=['relu', 'tanh', 'sigmoid'])
      ))
      model.add(Dropout(hp.Float('dropout_' + str(i), min_value=0.1, max_value=0.9, step=0.1)))

  model.add(Dense(1, activation='sigmoid'))
  model.compile(
      optimizer=hp.Choice('optimizer', values=['rmsprop', 'adam', 'sgd', 'nadam']),
      loss='binary_crossentropy',
      metrics=['accuracy']
  )

  return model

In [59]:
tuner = kt.RandomSearch(build_model , objective = 'val_accuracy' ,
                        max_trials = 3 ,
                        directory = "mydir" ,
                        project_name = 'final')

Reloading Tuner from mydir/final/tuner0.json


In [60]:
tuner.search(X_train, y_train, epochs=5, validation_data=(X_test, y_test))

In [61]:
tuner.get_best_hyperparameters()[0].values

{'num_layers': 7,
 'units_0': 88,
 'activation_0': 'tanh',
 'optimizer': 'rmsprop',
 'units_1': 72,
 'activation_1': 'relu',
 'units_2': 72,
 'activation_2': 'relu',
 'units_3': 8,
 'activation_3': 'relu',
 'units_4': 32,
 'activation_4': 'sigmoid',
 'units_5': 40,
 'activation_5': 'sigmoid',
 'units_6': 104,
 'activation_6': 'tanh',
 'units_7': 48,
 'activation_7': 'sigmoid',
 'units_8': 64,
 'activation_8': 'relu',
 'units_9': 128,
 'activation_9': 'sigmoid'}

In [62]:
model = tuner.get_best_models(num_models=1)[0]

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 2 variables whereas the saved optimizer has 18 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [63]:
model.fit(X_train , y_train , epochs=100 , initial_epoch=6 , validation_data=(X_test , y_test))

Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step - accuracy: 0.7573 - loss: 0.4763 - val_accuracy: 0.8117 - val_loss: 0.4210
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7980 - loss: 0.4361 - val_accuracy: 0.8377 - val_loss: 0.3962
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8241 - loss: 0.4193 - val_accuracy: 0.7987 - val_loss: 0.4216
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8225 - loss: 0.4117 - val_accuracy: 0.8117 - val_loss: 0.4147
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8404 - loss: 0.4001 - val_accuracy: 0.8182 - val_loss: 0.4236
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8469 - loss: 0.3882 - val_accuracy: 0.8182 - val_loss: 0.4141
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8436 - loss: 0.3652 - val_accuracy: 0.8182 - val_loss: 0.4092
Epoch 14/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.8583 - loss: 0.3666 - val_accuracy: 0.81